# Wavelength Censoring
Trying out different hyperparamters on robusta hmf

## Revision history
- last updated by Nana 2026-07-08

## Notes and bugs and to-dos:


In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robusta_hmf import Robusta
import time

In [ ]:
#define constants
c = 299792.458 #km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1
H_ALPHA, H_BETA = 6564.61, 4862.68

In [ ]:
#read in the data and deal with radial velocities
with open('hot_stars_subsample_2026-06-27_data.pkl','rb') as f:
     fluxes, loglam, ivars, continuua, spec_files, rvs = pickle.load(f)
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape, spec_files.shape)

In [ ]:
#log lambda values have been corrupted
print(10**loglam)
print(loglam[1:] - loglam[:-1])
print(np.median(loglam[1:] - loglam[:-1]))
delta_log_lambda_pixel = np.median(loglam[1:] - loglam[:-1])
wave = 10**loglam

In [ ]:
#get ready to do pixel shifts
delta_log_lambdas = (rvs / c) / LN10
delta_pixels = np.round(delta_log_lambdas/delta_log_lambda_pixel).astype(int)

print(delta_log_lambdas, delta_pixels)

In [ ]:
#shift spectra to the rest frame
## johanna is not going to like this

rest_fluxes = np.zeros_like(fluxes) + 1.
rest_ivars = np.zeros_like(ivars)
rest_continuua = np.zeros_like(continuua) + np.nan
for i, dp in enumerate(delta_pixels):
    if dp < 0:
        rest_fluxes[i, -dp:] = fluxes[i, :dp]
        rest_ivars[i, -dp:] = ivars[i, :dp]
        rest_continuua[i, -dp:] = continuua[i, :dp]
        
    elif dp > 0:
        rest_fluxes[i, :-dp] = fluxes[i, dp:]
        rest_ivars[i, :-dp] = ivars[i, dp:]
        rest_continuua[i, :-dp] = continuua[i, dp:]
        
    else:
        rest_fluxes[i, :] = fluxes[i, :]
        rest_ivars[i, :] = ivars[i, :]
        rest_continuua[i, :] = continuua[i, :]


In [ ]:
#fix nans and infinities
bad = np.logical_not(np.isfinite(rest_fluxes))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = np.logical_not(np.isfinite(rest_ivars))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = np.logical_or((rest_fluxes > 2.0), (rest_fluxes < 0))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = rest_ivars > MAX_IVAR
# rest_fluxes[bad] = 1
# rest_ivars[bad] = 0
rest_ivars[bad] = MAX_IVAR

bad = ivars < MIN_IVAR
rest_fluxes[bad] = 1.0
rest_ivars[bad] = MIN_IVAR

In [ ]:
#we are using the rest frame data as our data 
data = rest_fluxes
weights = rest_ivars

In [ ]:
## for nana:
#theory behind masking
#When a star has an emisison line from a disk around a star, the movement of the gas (towards or away from you) causes the
#emisison line to be more broad than "true" by the Doppler effect. We will be using a mask width  of 2000km/s (half-widthm of 1000km/s). 
#Doppler says  delta λ/λ* = v/c. Recall ln(1 + x) = x for small x, and ln(1+delta λ/λ* ) = v/c which is just delta_ln_λ. 
#lnλ diff is nice becasue boss pixels are uniform spaced in lnλ.

#so our width is delta_ln_λ = (1000km/s)/c. So for each wavelength that we care about (ask Johanna for this) just check that for every pixel
#within this width has weigths set to 0. 

In [ ]:
N, M = fluxes.shape
foo = np.arange(N)
rng = np.random.default_rng(17)
rng.shuffle(foo)
Aindx = foo[:N // 2]
Bindx = foo[N // 2:]

K = 12

modelA = Robusta(rank=K, robust=True, robust_scale=1, robust_nu=1)
start = time.perf_counter()
modelA.fit(data[Aindx], weights[Aindx], max_iter=10000)
end = time.perf_counter()
print("total time:", (end - start)/60, "minutes")

In [ ]:
#pseudo - ish code from hogg
# For the synthesising put a 2000km/s  (half-width of 1000) second region around the two wavelengths
# M points
# The width is delta_ln(wavelength) = (1000km/s)/c
# 0s at wavelength 
# Weights (N,M)
# Test_data (N,M)

# Test.synthmodel (test.data, weights * mask[None, :])


In [ ]:
ln_H_ALPHA, ln_H_BETA = np.log10(H_ALPHA), np.log10(H_BETA)

delta_lnlam = 1000/c
region = np.abs(loglam - ln_H_ALPHA)

censor_mask = np.ones_like(loglam)
censor_mask[region < delta_lnlam] = 0 #?

state, _ = modelA.infer(data[Bindx], weights[Bindx] * censor_mask[None, :])
synth_B = modelA.synthesize(state)
# mad = np.median(np.abs(data[Bindx] - synth_B))
# print(mad)
# residual = data[Bindx] - synth_B
# rms = np.sqrt(np.mean(residual ** 2))
# print(rms)

In [ ]:
print(loglam.shape)

In [ ]:
f = plt.figure(figsize=(15, 15))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[i]] + i * offset, color="k")
    f.gca().plot(wave, synth_B[i] + i * offset, color="r")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.xlabel("Wavelength")
plt.show()
#plt.savefig("K_12_fitted_example.png")